# NOTES 

- tried to find a list of flights and duration of the flights or datasets with time schedule, departure and arrival, but couldn't, same for rail
- paper with the exact same task as ours - https://ec.europa.eu/regional_policy/sources/work/2023-rail-vs-air_en.pdf

- what we have
    - AIR
        -  air travel estimate between airports, calculated with airport coordinates
        -  split into frame with direct flight and without
          
- what we need to do -
    - AIR
        - what about non-direct flights?
        - improve the air travel estimate - take into account speed of airplane routes, boarding time and leaving the airport time, getting to and from the airport time (with the city coordinates we already have in the dataframe)

    - RAIL 
        - selecting rail station in the city, take into account speed of trains between the given cities, boarding time and departure time
          
     
-----
- additionly notes - city names are fucked up, they are written differently in different data, so when we are going to be choosing the size of the city, we have to check if all the data is there, so far all cities above 500 000 have coordinates, but not all are assigned with the airport, probably will have to add some manually, also not all airport assigned have coordinates
- flight route database is 7 y old - kind of old
- data
- https://ec.europa.eu/eurostat/databrowser/view/urb_cpop1/default/table?lang=en&category=urb.urb_cgc
- https://public.opendatasoft.com/explore/dataset/geonames-all-cities-with-a-population-1000/export/?disjunctive.cou_name_en&sort=name&refine.timezone=Europe
- https://www.partow.net/miscellaneous/airportdatabase/
- the rest - from kaggle


## Narrowing down the scope
- transit points to the airport and station from a city center
    - for reducing workload, we set an estimate for the airport travel time of 1 hour to the station (assuming it to be near to the city center) 0.5 hour (2nd not agreed yet, suggestion Hedda),
    - Addition: research found that travellers spent an average of 157 minutes at airports and 32 minutes at railway stations (https://opentransportationjournal.com/VOLUME/13/PAGE/48/FULLTEXT/) adding that, too?
- considering only cities with population > 500.000
- considering only flights with direct connections

## Open Q's Hedda
- Can we add the countries to the cities df? In the source, they have the Codes indicating the countries
- What about cities that do not have an airport? Do we already have a solution for that?


In [10]:
import pandas as pd

# CITIES AND POPULATION SIZES 

# found this here https://ec.europa.eu/eurostat/databrowser/view/urb_cpop1/default/table?lang=en&category=urb.urb_cgc

# Load and process population data
file_path = '/kaggle/input/cities-and-population-sizes/new_file.xlsx'  
df = pd.read_excel(file_path, header=8)

# Rename the first column to "CITIES (Labels)"
df.rename(columns={df.columns[0]: "CITIES (Labels)"}, inplace=True)

# Replace any non-numeric population values (like ':', 'b', 'd', etc.) with NaN
df.replace({':': pd.NA, 'b': pd.NA, 'd': pd.NA, 'e': pd.NA, 'p': pd.NA}, inplace=True)

# Drop rows where the 'CITIES (Labels)' column is NaN or contains invalid values
df = df[~df['CITIES (Labels)'].isin(['Available flags:', 'Special value', ':', 'b', 'd', 'e', 'p'])]

# Find the index where 'Türkiye' is in the "CITIES (Labels)" column
turkiye_index = df[df['CITIES (Labels)'] == 'Türkiye'].index[0]

# Remove all rows up to and including the row with "Türkiye"
df = df.iloc[turkiye_index+1:]

# Ensure the columns with years are in numeric format, and drop any invalid values
df.iloc[:, 1:] = df.iloc[:, 1:].apply(pd.to_numeric, errors='coerce')

# Drop rows where the 'CITIES (Labels)' column is NaN
df = df[df['CITIES (Labels)'].notna()]

# Now proceed with finding the latest population for each city
latest_years = []
latest_populations = []

# Iterate over each row (city) to find the latest non-NaN population value and year
for index, row in df.iterrows():
    # Find the last non-NaN population value and its corresponding year
    latest_year = row.dropna().index[-1]  # Get the last non-NaN column (year)
    latest_population = row[latest_year]  # Get the population value for that year
    # Append the results to the lists
    latest_years.append(latest_year)
    latest_populations.append(latest_population)

# Create a new DataFrame with the city names, latest year, and latest population size
latest_df = pd.DataFrame({
    'City': df['CITIES (Labels)'],
    'Population': latest_populations
})
latest_df_sorted = latest_df.sort_values(by='Population', ascending=False)
latest_df_sorted = latest_df_sorted[latest_df_sorted['Population'] > 500000].sort_values(by='Population', ascending=False)

# Standardize cities in the latest_df_sorted
latest_df_sorted['City'] = latest_df_sorted['City'].str.strip().str.lower()

# Remove cities that are in the "entries_to_remove" list
entries_to_remove = [
    'kirklees', 'north lanarkshire (airdrie/bellshill/coatbridge/motherwell)',
    'cheshire west and chester', 'sandwell', 'wirral', 'brighton and hove',
    'medway', 'sefton', 'greater valletta', 'trafford', 'tameside', 'north tyneside'
]
latest_df_sorted = latest_df_sorted[~latest_df_sorted['City'].isin(entries_to_remove)]
latest_df_sorted.reset_index(drop=True, inplace=True)

# City replacements
city_replacements = {
    'bruxelles/brussel': 'brussels',
    'helsinki/helsingfors': 'helsinki',
    'lódz': 'lodz', 
    "'s-gravenhage": 'hague', 
    'alicante/alacant':'alicante',
    'pamplona/iruña':'pamplona',
    'espoo/esbo':'espoo',
    'vitoria/gasteiz':'vitoria',
    'tampere/tammerfors':'tampere', 
    'vantaa/vanda':'vantaa', 
    'oulu/uleåborg':'oulu', 
    'elche/elx':'elche',
    'turku/åbo':'turku', 
    'wien':'vienna', 
    'roma':'rome',
    'bucuresti':'bucarest', 
    'napoli':'naples',
    'praha':'prague', 
    'münchen':'munich'
}
latest_df_sorted['City'] = latest_df_sorted['City'].replace(to_replace=city_replacements)

print(latest_df_sorted)

         City  Population
0       paris  10306512.0
1    istanbul   9897599.0
2      london   8866541.0
3      madrid   5115272.0
4   barcelona   3767382.0
..        ...         ...
75  antwerpen    541122.0
76   bradford    535987.0
77   nürnberg    523026.0
78  edinburgh    515855.0
79   duisburg    502211.0

[80 rows x 2 columns]


In [11]:
# https://public.opendatasoft.com/explore/dataset/geonames-all-cities-with-a-population-1000/export/?disjunctive.cou_name_en&sort=name&refine.timezone=Europe - got the data here
# Load the city data for coordinates
city_data = pd.read_csv("/kaggle/input/coordinates/geonames-all-cities-with-a-population-1000.csv", delimiter=';')

# Clean columns in city_data
city_data.rename(columns={'Name': 'City'}, inplace=True)
city_data['City'] = city_data['City'].str.strip().str.lower()
city_data['ASCII Name'] = city_data['ASCII Name'].str.strip().str.lower()
city_data['Alternate Names'] = city_data['Alternate Names'].str.strip().str.lower()

# Merge latest_df_sorted with city_data for coordinates, ASCII Name, and Alternate Names
merged_df = pd.merge(
    latest_df_sorted,
    city_data[['City', 'Coordinates', 'ASCII Name', 'Alternate Names']],
    on='City',
    how='left'
)

# Function to find coordinates based on Alternate Names
def find_coordinates(city, city_data):
    # Check if the city is in 'ASCII Name'
    row = city_data[
        (city_data['ASCII Name'] == city)
    ]
    
    # If not found, check if the city is in 'Alternate Names'
    if row.empty:
        row = city_data[
            city_data['Alternate Names'].str.contains(fr'\b{city}\b', na=False)
        ]
    
    if not row.empty:
        return row.iloc[0]['Coordinates'], row.iloc[0]['ASCII Name'], row.iloc[0]['Alternate Names']
    return None, None, None

# Resolve missing coordinates and update the DataFrame
missing_coords = merged_df[merged_df['Coordinates'].isna()]
for index, row in missing_coords.iterrows():
    city = row['City']
    coordinates, ascii_name, alternate_names = find_coordinates(city, city_data)
    if coordinates:
        merged_df.loc[index, 'Coordinates'] = coordinates
        merged_df.loc[index, 'ASCII Name'] = ascii_name
        merged_df.loc[index, 'Alternate Names'] = alternate_names

# Split 'Coordinates' into 'CityLatitude' and 'CityLongitude'
if 'Coordinates' in merged_df.columns:
    merged_df[['CityLatitude', 'CityLongitude']] = merged_df['Coordinates'].str.split(',', expand=True)
    merged_df['CityLatitude'] = pd.to_numeric(merged_df['CityLatitude'], errors='coerce')
    merged_df['CityLongitude'] = pd.to_numeric(merged_df['CityLongitude'], errors='coerce')
    merged_df.drop(columns=['Coordinates'], inplace=True)

print(merged_df[['City', 'Population', 'CityLatitude', 'CityLongitude']])
# Check for rows with missing coordinates
missing_coordinates = merged_df[merged_df['CityLatitude'].isna() | merged_df['CityLongitude'].isna()]
if not missing_coordinates.empty:
    print("Rows with missing CityLatitude or CityLongitude:")
    print(missing_coordinates)
else:
    print("No missing coordinates found.")

         City  Population  CityLatitude  CityLongitude
0       paris  10306512.0      48.85341        2.34880
1    istanbul   9897599.0      41.01384       28.94966
2      london   8866541.0      51.50853       -0.12574
3      madrid   5115272.0      40.41650       -3.70256
4   barcelona   3767382.0      41.38879        2.15899
..        ...         ...           ...            ...
81   bradford    535987.0      53.79391       -1.75206
82   nürnberg    523026.0      49.45421       11.07752
83  edinburgh    515855.0      55.95206       -3.19648
84   duisburg    502211.0      50.81584        4.55533
85   duisburg    502211.0      51.43247        6.76516

[86 rows x 4 columns]
No missing coordinates found.


In [12]:
# Load airport data
airport_data = pd.read_csv(
    "/kaggle/input/european-airports-iata-codes/european_iatas_df.csv",
    header=None, usecols=[0, 1, 2]
)
airport_data.columns = ['City', 'Airport_Name', 'Airport_Code']
airport_data['City'] = airport_data['City'].str.strip().str.lower()
airport_data['Airport_Name'] = airport_data['Airport_Name'].str.strip()

# Ensure merged_df's columns are in a comparable format
merged_df['City'] = merged_df['City'].str.strip().str.lower()
merged_df['ASCII Name'] = merged_df['ASCII Name'].str.strip().str.lower()
merged_df['Alternate Names'] = merged_df['Alternate Names'].str.strip().str.lower()

# Add an empty column for the airport name
merged_df['Airport_Name'] = ''

# Match and add airport names
for _, row in airport_data.iterrows():
    airport_city = row['City']
    airport_name = row['Airport_Name']
    
    # Check if the airport city matches any city-related column
    merged_df.loc[
        (merged_df['City'] == airport_city) |
        (merged_df['ASCII Name'] == airport_city) |
        (merged_df['Alternate Names'].str.contains(fr'\b{airport_city}\b', na=False, regex=True)),
        'Airport_Name'
    ] += f"{airport_name}; "

# Clean up any trailing "; " from the Airport_Name column
merged_df['Airport_Name'] = merged_df['Airport_Name'].str.rstrip("; ")
#print(merged_df)

# change to airport per line
airport_names_split = merged_df['Airport_Name'].str.split(';', expand=True)
# Reshape the DataFrame to repeat other columns for each airport
# Use `melt` to convert wide format to long format
merged_df_repeated = merged_df.drop(columns='Airport_Name').join(airport_names_split)
# Step 3: Melt the DataFrame, keeping the 'City', 'Population', etc. columns intact
expanded_df = merged_df_repeated.melt(id_vars=merged_df.columns[:-1], value_name='Airport_Name')
# Step 4: Drop rows where 'Airport_Name' is NaN (empty cells after melting)
expanded_df = expanded_df.dropna(subset=['Airport_Name'])
expanded_df.reset_index(drop=True, inplace=True)
expanded_df = expanded_df.sort_values(by='Population', ascending=False)

expanded_df['Airport_Name'] = expanded_df['Airport_Name'].str.strip()
airport_data['Airport_Name'] = airport_data['Airport_Name'].str.strip()
# Merge on Airport_Name only
expanded_df = expanded_df.merge(
    airport_data[['Airport_Name', 'Airport_Code']],
    on='Airport_Name', 
    how='left'
)

# Print only the desired columns
#print(expanded_df[['City', 'Population', 'Airport_Name', 'Airport_Code']])

# Filter the rows without missing Airport_Name
expanded_df_no_missing_airport = expanded_df[expanded_df['Airport_Name'] != '']

# Print the desired columns of the cleaned dataframe
print(expanded_df_no_missing_airport[['City', 'Population', 'Airport_Name', 'Airport_Code']])

# Rows with no assigned airport code
print("")
no_airport_rows = expanded_df[expanded_df['Airport_Name'] == '']
print("Rows with missing Airport Name:")
print("")
print(no_airport_rows[['City', 'Population', 'Airport_Name', 'Airport_Code']])

          City  Population                         Airport_Name Airport_Code
0        paris  10306512.0            Charles de Gaulle Airport          CDG
1        paris  10306512.0                         Orly Airport          ORY
2     istanbul   9897599.0        Atatürk International Airport          IST
3     istanbul   9897599.0  Sabiha Gökçen International Airport          SAW
4       london   8866541.0                     Southend Airport          SEN
..         ...         ...                                  ...          ...
101     poznan    545073.0                Poznań Ławica Airport          POZ
102   hannover    545045.0                  Langenhagen Airport          HAJ
103  antwerpen    541122.0        Antwerp International Airport          ANR
105   nürnberg    523026.0               Albrecht Dürer Airport          NUE
106  edinburgh    515855.0                    Edinburgh Airport          EDI

[98 rows x 4 columns]

Rows with missing Airport Name:

           City  Po

In [13]:
# https://www.partow.net/miscellaneous/airportdatabase/ - found airport coordinates here
# Define a function to convert to decimal coordinates
def to_decimal(degrees, minutes, seconds, direction):
    decimal = float(degrees) + float(minutes) / 60 + float(seconds) / 3600
    if direction in ['S', 'W']:  # Negative for south and west
        decimal = -decimal
    return decimal

# Read the data from a text file
data = pd.read_csv("/kaggle/input/airportscoordinates/GlobalAirportDatabase.txt", sep=":", header=None, encoding='utf-8')

# Extract relevant columns: IATA Code, City, Latitude, Longitude
data_extracted = data[[1, 5, 6, 7, 8, 9, 10, 11, 12]].copy()  # Add .copy() to avoid the warning

# Convert Latitude and Longitude to decimal format using .loc to avoid SettingWithCopyWarning
data_extracted.loc[:, 'Latitude_Decimal'] = data_extracted.apply(
    lambda row: to_decimal(row[5], row[6], row[7], row[8]), axis=1
)
data_extracted.loc[:, 'Longitude_Decimal'] = data_extracted.apply(
    lambda row: to_decimal(row[9], row[10], row[11], row[12]), axis=1
)

# Select only the columns for IATA, City, Latitude, and Longitude
final_data = data_extracted[[1, 'Latitude_Decimal', 'Longitude_Decimal']]

# Rename columns to include Airport Code instead of IATA
final_data.columns = ['Airport_Code', 'AirportLatitude', 'AirportLongitude']
final_data = final_data.dropna(subset=['Airport_Code'])
final_data = final_data[(final_data['AirportLatitude'] != 0) & (final_data['AirportLongitude'] != 0)]

# Print the result
print(final_data)

     Airport_Code  AirportLatitude  AirportLongitude
0             GKA        -6.081667        145.391667
2             MAG        -5.206944        145.788611
3             HGU        -5.826111        144.296111
4             LAE        -6.569722        146.726111
5             POM        -9.443333        147.220000
...           ...              ...               ...
9275          KHG        39.543333         76.021667
9276          HTN        37.037778         79.865833
9278          URC        43.906944         87.474167
9286          HRB        45.623333        126.250278
9296          DLC        38.965556        121.538333

[2948 rows x 3 columns]


In [14]:
df = expanded_df_no_missing_airport.merge(final_data, how='left', on='Airport_Code')

# Print the result
print(df[['City', 'Population', 'Airport_Name', 'Airport_Code', 'CityLatitude', 'CityLongitude', 'AirportLatitude', 'AirportLongitude']])

# Filter rows where 'AirportLatitude' is NaN
print("")
print("Rows with missing Airport Coordinates:")
print("")
df_with_nan_latitude = df[df['AirportLatitude'].isna()]
print(df_with_nan_latitude[['City', 'Population', 'Airport_Name', 'Airport_Code', 'CityLatitude', 'CityLongitude', 'AirportLatitude', 'AirportLongitude']])

         City  Population                         Airport_Name Airport_Code  \
0       paris  10306512.0            Charles de Gaulle Airport          CDG   
1       paris  10306512.0                         Orly Airport          ORY   
2    istanbul   9897599.0        Atatürk International Airport          IST   
3    istanbul   9897599.0  Sabiha Gökçen International Airport          SAW   
4      london   8866541.0                     Southend Airport          SEN   
..        ...         ...                                  ...          ...   
95     poznan    545073.0                Poznań Ławica Airport          POZ   
96   hannover    545045.0                  Langenhagen Airport          HAJ   
97  antwerpen    541122.0        Antwerp International Airport          ANR   
98   nürnberg    523026.0               Albrecht Dürer Airport          NUE   
99  edinburgh    515855.0                    Edinburgh Airport          EDI   

    CityLatitude  CityLongitude  AirportLatitude  A

In [15]:
# HAVERSINE - ESTIMATING AIR DISTANCE

import math

def haversine(lat1, lon1, lat2, lon2):
    """Calculate great-circle distance between two points on the Earth."""
    R = 6371  # Earth radius in km
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat/2)**2 + math.cos(math.radians(lat1)) * math.cos(math.radians(lat2)) * math.sin(dlon/2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))
    return R * c

def air_travel_time(distance, speed=900, airport_time=2.5, local_transit_time=1):
    """Estimate total air travel time in hours."""
    flight_time = distance / speed
    return flight_time + airport_time + local_transit_time

def rail_travel_time(distance, speed=120, station_time=0.5):
    """Estimate total rail travel time in hours."""
    travel_time = distance / speed
    return travel_time + station_time

In [16]:
from itertools import combinations
import pandas as pd

# Generate city pairs from expanded_df, ensuring no duplicates
processed_pairs = set()  # Set to track processed pairs

city_pairs = [
    ((row1['City'], row1['AirportLatitude'], row1['AirportLongitude'], row1['Airport_Code']),
     (row2['City'], row2['AirportLatitude'], row2['AirportLongitude'], row2['Airport_Code']))
    for index1, row1 in df.iterrows()
    for index2, row2 in df.iterrows()
    if index1 < index2 and row1['City'] != row2['City']  # Ensure distinct cities
]

# Now filter out duplicate pairs
unique_city_pairs = []
for (city1, lat1, lon1, cod1), (city2, lat2, lon2, cod2) in city_pairs:
    sorted_pair = tuple(sorted([(city1, lat1, lon1, cod1), (city2, lat2, lon2, cod2)]))  # Sort cities to avoid reverse duplicates
    if sorted_pair not in processed_pairs:
        processed_pairs.add(sorted_pair)
        unique_city_pairs.append(((city1, lat1, lon1, cod1), (city2, lat2, lon2, cod2)))

# Calculate distances and air travel times
travel_data = []
for (city1, lat1, lon1, cod1), (city2, lat2, lon2, cod2) in unique_city_pairs:
    if not (pd.isna(lat1) or pd.isna(lon1) or pd.isna(lat2) or pd.isna(lon2)):
        distance = haversine(lat1, lon1, lat2, lon2)
        travel_time = air_travel_time(distance)
        travel_data.append((city1, lat1, lon1, cod1, city2, lat2, lon2, cod2, distance, travel_time))

# Create a dataframe with the results
travel_df = pd.DataFrame(travel_data, columns=['City1', 'AirportLat1', 'AirportLon1', 'Airport_Code1', 'City2', 'AirportLat2', 'AirportLon2', 'Airport_Code2', 'Distance (km)', 'Air Travel Time (h)'])
travel_df = travel_df.sort_values(by='Air Travel Time (h)', ascending=False)

# Display the dataframe
print("City Pairwise Air Travel Times:")
print(travel_df)

City Pairwise Air Travel Times:
           City1  AirportLat1  AirportLon1 Airport_Code1       City2  \
1623      lisboa    38.781111    -9.135833           LIS   gaziantep   
2989   gaziantep    36.948056    37.479167           GZT    bordeaux   
2857       porto    41.248056    -8.681389           OPO   gaziantep   
1616      lisboa    38.781111    -9.135833           LIS       adana   
1615      lisboa    38.781111    -9.135833           LIS       adana   
...          ...          ...          ...           ...         ...   
3678  düsseldorf    51.289444     6.766667           DUS    dortmund   
2795        köln    50.865833     7.142500           CGN  düsseldorf   
2532   rotterdam    51.957222     4.441667           RTM   amsterdam   
2473    brussels    50.902222     4.498611           BRU   antwerpen   
2866       porto    41.248056    -8.681389           OPO    bordeaux   

      AirportLat2  AirportLon2 Airport_Code2  Distance (km)  \
1623    36.948056    37.479167          

In [17]:
route_db = pd.read_csv('/kaggle/input/flight-route-database/routes.csv', header=None, low_memory=False)  # Load CSV without headers

# Set column names for the route database
route_db.columns = [
    'airline',              # Name of the airline
    'airline ID',           # Airline ID
    'source airport',       # Source airport code
    'source airport ID',    # Source airport ID
    'destination airport',  # Destination airport code
    'destination airport ID',  # Destination airport ID
    'codeshare',            # Codeshare information
    'stops',                # Number of stops
    'equipment'             # Equipment used
]
route_set = set(zip(route_db['source airport'], route_db['destination airport']))
travel_df['Route Exists'] = travel_df.apply(
    lambda row: (row['Airport_Code1'], row['Airport_Code2']) in route_set,
    axis=1
)

routes_exist_df = travel_df.loc[travel_df['Route Exists']]
routes_exist_not_df = travel_df.loc[travel_df['Route Exists'] == False]

print('Direct routes exist:')
print(f'Number of rows: {routes_exist_df.shape[0]}')
print(routes_exist_df[['City1', 'Airport_Code1', 'City2', 'Airport_Code2']])
print('')
print('')
print('Direct routes do not exist:')
print(f'Number of rows: {routes_exist_not_df.shape[0]}')
print(routes_exist_not_df[['City1', 'Airport_Code1', 'City2', 'Airport_Code2']])

Direct routes exist:
Number of rows: 1246
                  City1 Airport_Code1       City2 Airport_Code2
3550            antalya           AYT     glasgow           GLA
1643             lisboa           LIS    helsinki           HEL
3595           helsinki           HEL      málaga           AGP
190            istanbul           IST      lisboa           LIS
3559            antalya           AYT  manchester           MAN
...                 ...           ...         ...           ...
2839          amsterdam           AMS  düsseldorf           DUS
2431           brussels           BRU   amsterdam           AMS
3253  frankfurt am main           FRA   stuttgart           STR
3454             zürich           ZRH   stuttgart           STR
2205             munich           MUC    nürnberg           NUE

[1246 rows x 4 columns]


Direct routes do not exist:
Number of rows: 2548
           City1 Airport_Code1       City2 Airport_Code2
1623      lisboa           LIS   gaziantep           GZT


## Data Set Collection for Rail Travel Time

Methodology of paper:

1. *Station Selection* based on where time schedule data was available (I think they did it based on this MERITS database, not for free) (https://uic.org/passenger/passenger-services-group/merits#MERITS-FAQ):

- Selected stations in major urban centers
    - our definition of "urban center" for the time being would be cities above 500.000 population size
    - Kaggle dataset with train stations: https://www.kaggle.com/datasets/headsortails/train-stations-in-europe/data
      
- For cities with >10 stations, filtered to major stations based on:
    - Total departures between 6:00-20:00
    - Maximum length of direct trips
    - Station names
 
2. *Rail Travel Time Calculation:*

- Used OpenTripPlanner Analyst (how can we estimate that in an easier way OR use it, too?)
- https://docs.opentripplanner.org/en/latest/
- For me, it looks really complicated to use, but maybe you guys can have a look
    - Calculated best travel time every 15 minutes (4:00-23:45) on weekdays in 2019
        - Key parameters:
        - Max walking distance: 900m
        - Walking speed: 1.33 m/s
        - Maximum transfers: 5
        - Maximum trip time: 10 hours
        - Transit modes: rail, subway, walk
          
- Another idea would be to somehow scrape the travel time data, but I would not know how, and it also seems a bit too much effort. (what we need is GTFS data https://gtfs.org/getting-started/what-is-GTFS/; maybe here: https://www.transit.land/)
        
     
3. *TEN-T Network Analysis*:
- Here the actual TEN-T Map is available, and passenger railways, urban centers and even airports can be selected, but I don't know where to get the acutal data behind https://webgate.ec.europa.eu/tentec-maps/web/public/screen/home
- They talk about an API here (https://transport.ec.europa.eu/transport-themes/infrastructure-and-investment/trans-european-transport-network-ten-t/tentec-information-system-and-ten-t-map-library_en), but I cannot find it anywhere
- The place it should be would be here, but here they only have rather old data for port and airport: https://ec.europa.eu/eurostat/web/gisco/geodata/transport-networks
- I have requested access to the TENtec platfrom, hopefully I get access and the respective data is there

- Mapped connections to TEN-T rail network
    - Snapped city centroids to core network
    - Snapped to comprehensive network
    - Validated routes (network length ≤ 2.5× straight-line distance)
    - Prioritized core routes over comprehensive routes
    - Calculated network speeds and corrected outliers


4. Total Travel Time Calculation:
Sum of:
    - Car time from city center to departure station
    - Station transfer time (15 minutes)
    - Best available rail travel time
    - Car time from arrival station to destination city center
  
A really bad fallback solution would be to take the geodesic distance (straight line) and add the mean speed of the train. or manually input distances / times from researching ...

-> If we do not find a good solution until next week, we can ask in the review meeting.


Other potential sources:
- https://direkt.bahn.guru/?origin=8103000
    - they only use the API from DB: https://gist.github.com/juliuste/f9776a6b7925bc6cc2d52225dd83336e
    - this project also uses direct bahn guru: https://www.chronotrains.com/de/about
      
- https://www.openrailwaymap.org/
    - data on train routes from openstreetmap data: http://osmtrainroutes.bplaced.net/
 
- https://uic-stats.uic.org/select/ Here, a lot of data on rail is available, but I did not find anything for our specific topic



In [3]:
# kaggle dataset on train stations
df_trainstations = pd.read_csv('/kaggle/input/train-stations-in-europe/train_stations_europe.csv')

print("\n=== Dataset Info ===")
print(df_trainstations.info())

print("\n=== First few rows ===")
print(df_trainstations.head())


=== Dataset Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 64037 entries, 0 to 64036
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   id                 64037 non-null  int64  
 1   name               64036 non-null  object 
 2   name_norm          64036 non-null  object 
 3   uic                23487 non-null  float64
 4   latitude           62142 non-null  float64
 5   longitude          62142 non-null  float64
 6   parent_station_id  4014 non-null   float64
 7   country            64037 non-null  object 
 8   time_zone          64037 non-null  object 
 9   is_city            64037 non-null  bool   
 10  is_main_station    64037 non-null  bool   
 11  is_airport         64037 non-null  bool   
 12  entur_id           635 non-null    object 
 13  entur_is_enabled   64037 non-null  bool   
dtypes: bool(4), float64(4), int64(1), object(5)
memory usage: 5.1+ MB
None

=== First few rows ===
 

In [ ]:
# clean df, 
# match cities to the stations,
# filter df according to the cities >= 500.000

# Open Street Map Data via osmnx

In [2]:
!pip install osmnx

In [6]:
import osmnx as ox
import networkx as nx
import numpy as np
from shapely.geometry import Point, LineString, Polygon, box
from itertools import combinations
import pickle
import os
from datetime import datetime
import pandas as pd

In [4]:
class RailConnectionAnalyzer:
    def __init__(self, cities_df):
        """
        Initialize with DataFrame of European cities.
        
        Parameters:
        cities_df: DataFrame with columns:
            - City: Name of the city
            - Population: City population
            - CityLatitude: City latitude
            - CityLongitude: City longitude
        """
        # Clean city names and remove duplicates
        cities_df = cities_df.drop_duplicates(subset=['City'])
        self.cities = cities_df[cities_df['Population'] > 500000].copy()
        self.G = None
        self.connections = None
        self.cache_dir = 'rail_cache'
        
        if not os.path.exists(self.cache_dir):
            os.makedirs(self.cache_dir)
    
    def _get_cache_filename(self, bbox_buffer):
        """Generate a unique cache filename based on cities and buffer."""
        # Use hash of sorted city names to create unique identifier
        cities_str = '_'.join(sorted(self.cities['City']))
        return os.path.join(self.cache_dir, f"european_rail_{hash(cities_str)}_{bbox_buffer}.pkl")
    
    def get_railway_network(self, bbox_buffer=0.05, use_cache=True):
        """Get railway network, using cached data if available."""
        cache_file = self._get_cache_filename(bbox_buffer)
        
        if use_cache and os.path.exists(cache_file):
            print("Loading network from cache...")
            try:
                with open(cache_file, 'rb') as f:
                    self.G = pickle.load(f)
                print("Network loaded from cache successfully!")
                return self.G
            except Exception as e:
                print(f"Error loading cache: {str(e)}")
                print("Will fetch fresh data instead.")
        
        print("Fetching network data from OSM...")
        
        min_lat = self.cities['CityLatitude'].min() - bbox_buffer
        max_lat = self.cities['CityLatitude'].max() + bbox_buffer
        min_lon = self.cities['CityLongitude'].min() - bbox_buffer
        max_lon = self.cities['CityLongitude'].max() + bbox_buffer
        
        bbox_poly = box(min_lon, min_lat, max_lon, max_lat)
        
        custom_filter = (
            '["railway"="rail"]'
            '["service"!~"siding|yard|spur"]'
        )
        
        try:
            self.G = ox.graph_from_polygon(
                polygon=bbox_poly,
                custom_filter=custom_filter,
                simplify=True
            )
            
            if self.G is None or len(self.G) == 0:
                raise ValueError("No railway network found. Try increasing bbox_buffer.")
            
            print("Saving network to cache...")
            with open(cache_file, 'wb') as f:
                pickle.dump(self.G, f)
            print("Network cached successfully!")
                
        except Exception as e:
            print(f"Error fetching railway network: {str(e)}")
            raise
        
        return self.G
    
    def parse_speed(self, speed_value, default_speed=120):
        """Parse speed value from OSM data."""
        if isinstance(speed_value, (int, float)):
            return float(speed_value)
        
        if isinstance(speed_value, list):
            speeds = []
            for speed in speed_value:
                try:
                    num = ''.join(filter(str.isdigit, str(speed)))
                    if num:
                        speeds.append(float(num))
                except (ValueError, TypeError):
                    continue
            return max(speeds) if speeds else default_speed
        
        if isinstance(speed_value, str):
            try:
                num = ''.join(filter(str.isdigit, speed_value))
                return float(num) if num else default_speed
            except (ValueError, TypeError):
                return default_speed
        
        return default_speed
    
    def calculate_connections(self, default_speed=120):
        """Calculate rail connections between city pairs."""
        if self.G is None:
            raise ValueError("Network not loaded. Call get_railway_network first.")
        
        print(f"Calculating connections for {len(self.cities)} cities...")
        connections = []
        city_pairs = list(combinations(self.cities.itertuples(), 2))
        total_pairs = len(city_pairs)
        
        for i, (city1, city2) in enumerate(city_pairs, 1):
            print(f"Processing pair {i}/{total_pairs}: {city1.City} - {city2.City}")
            try:
                node1 = ox.nearest_nodes(self.G, city1.CityLongitude, city1.CityLatitude)
                node2 = ox.nearest_nodes(self.G, city2.CityLongitude, city2.CityLatitude)
                
                try:
                    path = nx.shortest_path(self.G, node1, node2, weight='length')
                except nx.NetworkXNoPath:
                    print(f"No path found between {city1.City} and {city2.City}")
                    connections.append(self._create_null_connection(city1.City, city2.City))
                    continue
                
                distance = 0
                time_hours = 0
                
                for j in range(len(path)-1):
                    u, v = path[j], path[j+1]
                    edge_data = self.G[u][v][0]
                    length_km = edge_data.get('length', 0) / 1000
                    max_speed = self.parse_speed(edge_data.get('maxspeed', default_speed), default_speed)
                    
                    distance += length_km
                    time_hours += length_km / max_speed
                
                connections.append({
                    'city1': city1.City,
                    'city2': city2.City,
                    'distance_km': round(distance, 2),
                    'estimated_time_hours': round(time_hours, 2),
                    'average_speed': round(distance / time_hours if time_hours > 0 else 0, 2)
                })
                
            except Exception as e:
                print(f"Error processing connection between {city1.City} and {city2.City}: {str(e)}")
                connections.append(self._create_null_connection(city1.City, city2.City))
        
        self.connections = pd.DataFrame(connections)
        return self.connections
    
    def _create_null_connection(self, city1, city2):
        """Create a null connection entry."""
        return {
            'city1': city1,
            'city2': city2,
            'distance_km': None,
            'estimated_time_hours': None,
            'average_speed': None
        }

    def export_connections(self, filename='european_rail_connections.csv'):
        """Export connections to CSV file."""
        if self.connections is not None:
            self.connections.to_csv(filename, index=False)
            print(f"Connections exported to {filename}")

In [7]:
# test code because with our cities df it runs very long
# Test data - small subset of European cities
test_cities = pd.DataFrame({
    'City': ['Paris', 'Berlin', 'Munich', 'Frankfurt'],
    'Population': [10306512, 3669495, 1471508, 753056],
    'CityLatitude': [48.85341, 52.52001, 48.13743, 50.11052],
    'CityLongitude': [2.34880, 13.40495, 11.57549, 8.68417]
})

# 1. Basic test - create analyzer and get network
print("Test 1: Basic Network Retrieval")
analyzer = RailConnectionAnalyzer(test_cities)
G = analyzer.get_railway_network(bbox_buffer=0.05, use_cache=True)
connections = analyzer.calculate_connections()
print("\nInitial connections:")
print(connections)

Test 1: Basic Network Retrieval
Fetching network data from OSM...


/opt/conda/lib/python3.10/site-packages/osmnx/_overpass.py:265: UserWarning: This area is 158 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


Saving network to cache...
Network cached successfully!
Calculating connections for 4 cities...
Processing pair 1/6: Paris - Berlin
Processing pair 2/6: Paris - Munich
Processing pair 3/6: Paris - Frankfurt
Processing pair 4/6: Berlin - Munich
Processing pair 5/6: Berlin - Frankfurt
Processing pair 6/6: Munich - Frankfurt

Initial connections:
    city1      city2  distance_km  estimated_time_hours  average_speed
0   Paris     Berlin      1057.65                  7.82         135.20
1   Paris     Munich       846.53                  4.44         190.82
2   Paris  Frankfurt       587.91                  3.28         179.36
3  Berlin     Munich       632.59                  3.61         175.38
4  Berlin  Frankfurt       520.40                  3.45         150.84
5  Munich  Frankfurt       397.57                  2.40         165.50


In [18]:
# Create analyzer with your data
analyzer = RailConnectionAnalyzer(merged_df)

# Get network (will be cached for future use)
G = analyzer.get_railway_network(bbox_buffer=0.05, use_cache=True)

# Calculate connections
connections = analyzer.calculate_connections()

# Save results
analyzer.export_connections('european_rail_connections.csv')

# Display summary
print("\nConnection Summary:")
print(f"Total connections calculated: {len(connections)}")
print("\nSample of connections:")
print(connections.head())

Fetching network data from OSM...


/opt/conda/lib/python3.10/site-packages/osmnx/_overpass.py:265: UserWarning: This area is 3,689 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


KeyboardInterrupt: 

# test code because